# TMJDLC — Pyrogram Telegram Bot (Google Colab)

**Runtime:** Pyrogram MTProto handles bot updates, commands, callbacks, and Telegram media uploads. Neon PostgreSQL stores the durable job queue, stage receipts, worker leases, and destination channel settings.

Setup is split into separate steps: sync source, install dependencies, configure secrets, preflight, then start. After the first installation in a fresh Colab runtime, you do not need to reinstall packages every time you restart the bot.


## 1. Configure Colab Secrets

Add these five secrets in Colab's Secrets panel and enable notebook access:

- `TG_BOT_TOKEN` — token from @BotFather
- `TG_API_ID` — Telegram API ID from my.telegram.org
- `TG_API_HASH` — Telegram API hash from my.telegram.org
- `TG_ADMIN_IDS` — numeric Telegram user ID(s); separate multiple IDs with commas
- `DATABASE_URL` — Neon PostgreSQL connection string

The destination video and screenshot channel IDs are configured later with `/setup` and saved in Neon. The bot must be an administrator in each destination channel.


In [ ]:
#@title 2. Sync latest TMJDLC source
%cd /content
import os, subprocess
repo = '/content/TMJDLC'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/Telebotfaroff/TMJDLC.git', repo], check=True)
%cd /content/TMJDLC
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'TMJDLC source synced: {commit}')


In [ ]:
#@title 3. Install dependencies (run once per fresh Colab runtime)
%cd /content/TMJDLC
import subprocess, sys
print('Installing Python package and dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Installing FFmpeg...')
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y ffmpeg aria2'], check=True)
print('Installing Playwright Chromium...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'tmjdlc'], check=True)
print('Installation and Python compile check completed.')


In [ ]:
#@title 4. Load credentials and set worker options
,import os
,from google.colab import userdata
,SECRET_NAMES = ['TG_BOT_TOKEN', 'TG_API_ID', 'TG_API_HASH', 'TG_ADMIN_IDS', 'DATABASE_URL']
,missing = []
,for key in SECRET_NAMES:
,    try:
,        value = userdata.get(key)
,    except Exception:
,        value = None
,    if not value:
,        missing.append(key)
,    else:
,        os.environ[key] = str(value).strip()
,if missing:
,    raise RuntimeError('Missing Colab Secrets: ' + ', '.join(missing))
,
,MAX_PARALLEL = 2 #@param {type:'slider', min:1, max:4, step:1}
,UPLOAD_WORKERS = 4 #@param {type:'slider', min:1, max:8, step:1}
,DOWNLOAD_FRAGMENTS = 8 #@param {type:'slider', min:1, max:16, step:1}
,MIN_FREE_DISK_GB = 3.0 #@param {type:'number'}
,AUTO_CLEAN = True #@param {type:'boolean'}
,NOTIFY_PROGRESS_STEP = 10 #@param {type:'integer'}
,QUEUE_POLL_SECONDS = 4 #@param {type:'integer'}
,JOB_MAX_ATTEMPTS = 3 #@param {type:'integer'}
,JOB_LEASE_SECONDS = 3600 #@param {type:'integer'}
,os.environ['TMJDLC_MAX_PARALLEL'] = str(MAX_PARALLEL)
,os.environ['TMJDLC_UPLOAD_WORKERS'] = str(UPLOAD_WORKERS)
,os.environ['TMJDLC_DOWNLOAD_FRAGMENTS'] = str(DOWNLOAD_FRAGMENTS)
os.environ['TMJDLC_USE_ARIA2'] = 'true'
,os.environ['TMJDLC_MIN_FREE_DISK_GB'] = str(MIN_FREE_DISK_GB)
,os.environ['TMJDLC_AUTO_CLEAN'] = str(AUTO_CLEAN).lower()
,os.environ['TMJDLC_NOTIFY_PROGRESS_STEP'] = str(NOTIFY_PROGRESS_STEP)
,os.environ['TMJDLC_QUEUE_POLL_SECONDS'] = str(QUEUE_POLL_SECONDS)
,os.environ['TMJDLC_JOB_MAX_ATTEMPTS'] = str(JOB_MAX_ATTEMPTS)
,os.environ['TMJDLC_JOB_LEASE_SECONDS'] = str(JOB_LEASE_SECONDS)
,os.environ['TMJDLC_WORK_DIR'] = '/content/tmjdlc-bot'
,os.makedirs(os.environ['TMJDLC_WORK_DIR'], exist_ok=True)
,print('Credentials loaded (values hidden).')
,print(f'Workers: {MAX_PARALLEL} | Upload transmissions: {UPLOAD_WORKERS} | Download fragments: {DOWNLOAD_FRAGMENTS}')
,print(f'Pause below: {MIN_FREE_DISK_GB} GiB free | Retry limit: {JOB_MAX_ATTEMPTS}')
,print(f'Auto-clean: {AUTO_CLEAN} | Telegram progress interval: {NOTIFY_PROGRESS_STEP}%')
,

In [ ]:
#@title 5. Preflight checks
import shutil, importlib.util
for executable in ('ffmpeg', 'ffprobe'):
    if shutil.which(executable) is None:
        raise RuntimeError(f'{executable} is not installed or not on PATH.')
for module in ('pyrogram', 'psycopg', 'tmjdlc'):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f'Required Python module is missing: {module}')
disk = shutil.disk_usage(os.environ['TMJDLC_WORK_DIR'])
print(f'FFmpeg/FFprobe: ready | Free disk: {disk.free / 1024**3:.1f} GiB')
print('Preflight passed. Neon schema initialization and expired-job recovery run at bot startup.')


## 6. Start the bot

Run the cell below and keep the Colab runtime alive while jobs are processing. Stop it with the notebook stop button or Runtime → Interrupt execution.

On startup, the bot initializes the Neon tables and recovers jobs whose worker leases expired. Downloaded videos and screenshots live on Colab's temporary disk; Neon stores job state and stage receipts, not the media files. If Colab disconnects, files on its local disk may be lost, so a retry may need to download the media again.


In [ ]:
#@title 7. Start TMJDLC — Pyrogram MTProto
%cd /content/TMJDLC
!bash -o pipefail -c 'PYTHONUNBUFFERED=1 python -u -m tmjdlc.bot 2>&1 | tee -a /content/tmjdlc-bot/bot.log'


## Telegram commands

- `/start` — command help
- `/setup` — guided video and screenshot destination setup (saved in Neon)
- `/setupload CHANNEL_ID` / `/setimages CHANNEL_ID` — set destinations directly
- `/dl VIDEO_URL` — queue one Javtiful video
- `/batch CHANNEL_URL all` — queue all channel pages
- `/batch CHANNEL_URL 1-5,8` — queue selected pages
- `/panel` — queue dashboard and recent jobs
- `/queue` — queue counts and recent jobs
- `/settings` — current destination and worker settings
- `/storage` — disk usage and download pause threshold
- `/cancel JOB_ID` — cancel queued or active work
- `/retry JOB_ID` — retry a failed job

Use `/setup` once before queueing jobs. Make sure the bot is an administrator in both configured channels. Worker count is limited to 1–4; increase it only when the Colab runtime has enough CPU, RAM, network, and disk capacity.
